# Computer Vision Quiz Practice (Criteria Focus)

Use this exact flow to maximize score:

**List -> Load -> Smooth -> Detect -> Match -> Filter -> Draw -> Show**

## Rubric Mapping
- Preprocess (30): Listing image (10), Load image (10), Smoothing image (10)
- Feature detect (15)
- Feature matching (35): Match (15), Filter match (20)
- Result (20): Draw match (10), Show match (10)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# === PREPROCESS: LISTING IMAGE (10) ===
BASE_DIR = Path('CV_Session6/Dataset')
OBJECT_PATH = BASE_DIR / 'Object.jpg'
DATA_DIR = BASE_DIR / 'Data'

if not OBJECT_PATH.exists():
    raise FileNotFoundError(f'Target image not found: {OBJECT_PATH}')
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Data folder not found: {DATA_DIR}')

valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
candidate_paths = sorted([p for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() in valid_ext])

print(f'Target image: {OBJECT_PATH}')
print(f'Candidate images listed: {len(candidate_paths)}')
print('First 5 candidates:')
for p in candidate_paths[:5]:
    print(' -', p.name)

In [ ]:
# === PREPROCESS: LOAD IMAGE (10) ===
target_bgr = cv2.imread(str(OBJECT_PATH))
if target_bgr is None:
    raise ValueError(f'Failed to load target image: {OBJECT_PATH}')

candidates = []
for p in candidate_paths:
    img = cv2.imread(str(p))
    if img is not None:
        candidates.append((p, img))

print(f'Loaded target shape: {target_bgr.shape}')
print(f'Loaded candidates: {len(candidates)}')

target_rgb = cv2.cvtColor(target_bgr, cv2.COLOR_BGR2RGB)
plt.imshow(target_rgb)
plt.title('Target Image')
plt.axis('off')
plt.show()

In [ ]:
# === PREPROCESS: SMOOTHING IMAGE (10) ===
def preprocess_for_features(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    smooth = cv2.medianBlur(gray, 5)
    return smooth

target_gray = preprocess_for_features(target_bgr)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(target_rgb)
plt.title('Original (RGB)')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(target_gray, cmap='gray')
plt.title('Gray + Median Blur')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# === FEATURE DETECT (15) ===
# Prefer SIFT to match your Session 6 pipeline, fallback to ORB if SIFT is unavailable.
if hasattr(cv2, 'SIFT_create'):
    detector_name = 'SIFT'
    detector = cv2.SIFT_create(nfeatures=5000, contrastThreshold=0.03, edgeThreshold=15)
    norm_type = cv2.NORM_L2
    use_flann = True
else:
    detector_name = 'ORB'
    detector = cv2.ORB_create(nfeatures=3000)
    norm_type = cv2.NORM_HAMMING
    use_flann = False

target_kp, target_desc = detector.detectAndCompute(target_gray, None)
if target_desc is None or len(target_kp) < 2:
    raise RuntimeError('Target descriptors are not enough for matching.')

print(f'Detector: {detector_name}')
print(f'Target keypoints: {len(target_kp)}')

In [ ]:
# === FEATURE MATCHING (35): MATCH (15) + FILTER MATCH (20) ===
if use_flann:
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=200)
    matcher = cv2.FlannBasedMatcher(index_params, search_params)
    target_desc_for_match = np.float32(target_desc)
else:
    matcher = cv2.BFMatcher(norm_type)
    target_desc_for_match = target_desc

best = None
ratio_threshold = 0.75

for path, img_bgr in candidates:
    gray = preprocess_for_features(img_bgr)
    kp, desc = detector.detectAndCompute(gray, None)

    if desc is None or len(kp) < 2:
        continue

    desc_for_match = np.float32(desc) if use_flann else desc
    knn_matches = matcher.knnMatch(target_desc_for_match, desc_for_match, k=2)

    match_mask = [[0, 0] for _ in range(len(knn_matches))]
    good_count = 0

    # Lowe ratio test: keep only reliable matches.
    for i, pair in enumerate(knn_matches):
        if len(pair) < 2:
            continue
        fm, sm = pair
        if fm.distance < ratio_threshold * sm.distance:
            match_mask[i] = [1, 0]
            good_count += 1

    if best is None or good_count > best['good_count']:
        best = {
            'path': path,
            'img_bgr': img_bgr,
            'kp': kp,
            'matches': knn_matches,
            'mask': match_mask,
            'good_count': good_count,
        }

if best is None:
    raise RuntimeError('No candidate produced valid matches.')

print(f'Best candidate: {best["path"].name}')
print(f'Good matches (after filtering): {best["good_count"]}')

In [ ]:
# === RESULT (20): DRAW MATCH (10) + SHOW MATCH (10) ===
best_rgb = cv2.cvtColor(best['img_bgr'], cv2.COLOR_BGR2RGB)

draw_params = dict(
    matchColor=(0, 255, 0),
    singlePointColor=(255, 0, 0),
    matchesMask=best['mask'],
    flags=cv2.DrawMatchesFlags_DEFAULT,
)

result_img = cv2.drawMatchesKnn(
    target_rgb, target_kp,
    best_rgb, best['kp'],
    best['matches'], None, **draw_params
)

plt.figure(figsize=(22, 10))
plt.imshow(result_img)
plt.title(f'Best Match: {best["path"].name} | Good Matches: {best["good_count"]}')
plt.axis('off')
plt.show()

## 20-Second Oral Recall
1. **Listing**: `os.listdir` / `Path.iterdir` and extension filter.
2. **Load**: `cv2.imread` + color conversion.
3. **Smooth**: `cv2.medianBlur` (or Gaussian/Bilateral).
4. **Detect**: `SIFT_create` + `detectAndCompute`.
5. **Match**: `FlannBasedMatcher` + `knnMatch(k=2)`.
6. **Filter**: Lowe ratio test `m.distance < 0.75 * n.distance`.
7. **Result**: `cv2.drawMatchesKnn` + `plt.imshow`.